# DIU Splicing Event Classification

Classify alternative splicing event types (SE, RI, A3SS, A5SS, MXE, AFE, ALE) for differentially-used isoforms from DEXSeq pairwise cell-type comparisons.

**Strategy**: for each gene × comparison, enumerate all pairs of significant isoforms and run the Wang bubble classifier. Results are aggregated into per-comparison event-type tallies.

In [ ]:
#| default_exp diu_events

In [ ]:
#| export
from __future__ import annotations
from itertools import combinations
from typing import Optional
import pandas as pd
from allos.transcript_data import TranscriptData, classify_transcript_pair_from_index

## Core classification function

In [ ]:
#| export
def classify_diu_events(
    td: TranscriptData,
    diu_df: pd.DataFrame,
    tid_col: str = "featureID",
    gene_col: str = "groupID",
    comparison_col: Optional[str] = "pair_id",
    lfc_col: Optional[str] = None,
) -> pd.DataFrame:
    """
    Classify splicing events for all pairs of differentially used isoforms.

    For each (comparison, gene) group, every unique pair of isoforms is run
    through the Wang bubble classifier. Isoforms not found in the TranscriptData
    index are silently skipped.

    Parameters
    ----------
    td : TranscriptData
        Loaded from the matched GTF.
    diu_df : pd.DataFrame
        Significant isoform rows from DEXSeq (one isoform per row).
    tid_col : str
        Column holding transcript / isoform IDs.
    gene_col : str
        Column holding gene name / groupID.
    comparison_col : str or None
        Column holding the comparison label (e.g. "Basal__vs__Multiciliated").
        Pass None if the DataFrame is already for a single comparison.
    lfc_col : str or None
        If given, the sign of lfc determines which isoform is labelled tid1
        (positive lfc → tid1 is higher in condition B).

    Returns
    -------
    pd.DataFrame
        Columns: comparison, gene, tid1, tid2, event, coordinates
    """
    idx = td._idx

    # Build grouping keys
    if comparison_col and comparison_col in diu_df.columns:
        group_keys = [comparison_col, gene_col]
    else:
        diu_df = diu_df.copy()
        diu_df["_comparison"] = "single"
        group_keys = ["_comparison", gene_col]
        comparison_col = "_comparison"

    rows = []
    for (comparison, gene), grp in diu_df.groupby(group_keys, sort=False):
        isoforms = grp[tid_col].unique().tolist()
        if len(isoforms) < 2:
            continue

        for tid1, tid2 in combinations(isoforms, 2):
            # Order by lfc if available so tid1 is the 'higher in B' isoform
            if lfc_col:
                lfc1 = grp.loc[grp[tid_col] == tid1, lfc_col].values
                lfc2 = grp.loc[grp[tid_col] == tid2, lfc_col].values
                if len(lfc1) and len(lfc2) and float(lfc1[0]) < float(lfc2[0]):
                    tid1, tid2 = tid2, tid1

            try:
                events = td.classify_splicing_events(tid1, tid2)
            except Exception:
                # transcript not in index or on different chroms/strands
                continue

            for _, ev_row in events.iterrows():
                rows.append({
                    "comparison": comparison,
                    "gene": gene,
                    "tid1": tid1,
                    "tid2": tid2,
                    "event": ev_row["event"],
                    "coordinates": ev_row["coordinates"],
                })

    return pd.DataFrame(rows, columns=["comparison", "gene", "tid1", "tid2", "event", "coordinates"])

## Summary helpers

In [ ]:
#| export
def event_type_summary(
    classified: pd.DataFrame,
    normalise: bool = True,
) -> pd.DataFrame:
    """
    Pivot classified events to a (comparison × event_type) count / proportion table.

    Parameters
    ----------
    classified : pd.DataFrame
        Output of classify_diu_events.
    normalise : bool
        If True, return row proportions (sum to 1 across event types).

    Returns
    -------
    pd.DataFrame
        Index = comparison, columns = event types.
    """
    tbl = (
        classified
        .groupby(["comparison", "event"])
        .size()
        .unstack("event", fill_value=0)
    )
    if normalise:
        tbl = tbl.div(tbl.sum(axis=1), axis=0)
    return tbl


def gene_event_summary(
    classified: pd.DataFrame,
) -> pd.DataFrame:
    """
    For each (comparison, gene) return the dominant event type and
    a count of all events detected.
    """
    def _dominant(grp):
        vc = grp["event"].value_counts()
        return pd.Series({
            "n_pairs": len(grp[["tid1","tid2"]].drop_duplicates()),
            "n_events": len(grp),
            "dominant_event": vc.index[0],
            "event_counts": vc.to_dict(),
        })

    return (
        classified
        .groupby(["comparison", "gene"], sort=False)
        .apply(_dominant)
        .reset_index()
    )

---
## Demo — Discovair epithelial pairwise comparisons

### Identify the LFC column

DEXSeq names it `log2fold_{B}_{A}` — we detect it automatically.

### Run classification

This iterates over every (comparison × gene) group, creates all isoform pairs, and classifies each. Pairs where either transcript is absent from the index are silently skipped.

### Per-comparison event-type proportions

### Per-gene dominant events

### Visualise — stacked bar of event types per comparison

### Save outputs

---
## IsoformSwitchAnalyzeR-style switch classification

For each gene × comparison, computes isoform fractions in each condition, identifies the isoform with the **largest gain** in condition B and the isoform with the **largest gain** in condition A, then classifies that single pair. Requires loading the **all-features** file (not just significant isoforms) so fractions sum correctly across all isoforms of a gene.

In [ ]:
#| export
def classify_switch_isf(
    td: "TranscriptData",
    all_features_df: pd.DataFrame,
    sig_genes: set = None,
    tid_col: str = "featureID_bare",
    gene_col: str = "groupID",
    comparison_col: str = "pair_id",
    celltype_a_col: str = "celltype_A",
    celltype_b_col: str = "celltype_B",
    dif_cutoff: float = 0.1,
) -> pd.DataFrame:
    """
    IsoformSwitchAnalyzeR-style switch classification.

    For each gene × comparison:
      1. Compute isoform fractions in condition A and B from all isoforms.
      2. Compute delta isoform fraction (dIF = frac_B - frac_A).
      3. Pair the isoform with the largest positive dIF (gained in B)
         against the isoform with the largest negative dIF (gained in A).
      4. Classify that single pair with the bubble classifier.

    Parameters
    ----------
    all_features_df : pd.DataFrame
        All isoform rows (significant + non-significant) for correct fractions.
    sig_genes : set of (comparison, gene) tuples, optional
        If provided, restrict to genes with at least one significant isoform.
    dif_cutoff : float
        Minimum absolute delta isoform fraction to call a switch (default 0.1).
    """
    rows = []

    for (comparison, gene), grp in all_features_df.groupby(
        [comparison_col, gene_col], sort=False
    ):
        if sig_genes is not None and (comparison, gene) not in sig_genes:
            continue
        if len(grp) < 2:
            continue

        ct_a = grp[celltype_a_col].iloc[0]
        ct_b = grp[celltype_b_col].iloc[0]
        if ct_a not in grp.columns or ct_b not in grp.columns:
            continue

        expr_a = grp[ct_a].fillna(0).values.astype(float)
        expr_b = grp[ct_b].fillna(0).values.astype(float)
        total_a, total_b = expr_a.sum(), expr_b.sum()
        if total_a == 0 or total_b == 0:
            continue

        dif = (expr_b / total_b) - (expr_a / total_a)

        gained_idx = int(dif.argmax())   # up in B
        lost_idx   = int(dif.argmin())   # up in A

        if gained_idx == lost_idx:
            continue
        if abs(dif[gained_idx]) < dif_cutoff and abs(dif[lost_idx]) < dif_cutoff:
            continue

        tid_up_b = grp[tid_col].iloc[gained_idx]
        tid_up_a = grp[tid_col].iloc[lost_idx]

        try:
            events = td.classify_splicing_events(tid_up_b, tid_up_a)
        except Exception:
            continue

        for _, ev_row in events.iterrows():
            rows.append({
                "comparison":  comparison,
                "gene":        gene,
                "tid_up_in_B": tid_up_b,
                "tid_up_in_A": tid_up_a,
                "dIF_up_B":    round(float(dif[gained_idx]), 4),
                "dIF_up_A":    round(float(-dif[lost_idx]),  4),
                "event":       ev_row["event"],
                "coordinates": ev_row["coordinates"],
            })

    return pd.DataFrame(rows)


### Demo — ISF-style across all comparisons

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()